In [ ]:
import numpy as np
import sklearn.decomposition as dp
import pickle
import sys,os
import numpy.random as rand
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import auc,roc_curve,roc_auc_score
from sklearn.metrics import average_precision_score,precision_recall_curve
from sklearn.utils.random import sample_without_replacement


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
sys.path.append('/home/austin/DataAnalysis')
from data_tools import load_data


In [ ]:
fnm='/media/austin/ThickBoy__1/DataAgression_Granger2/Aggression_sub_12.mat'
power,coherence,granger,labels = load_data(fnm,fBounds=(1,56),
                        feature_list=['power','coherence','granger'])


In [ ]:
myLabel = labels['windows']
mouse = np.asarray(myLabel['mouse'])
group = np.asarray(myLabel['group'])
expDate = np.asarray(myLabel['expDate'])
behavior = np.asarray(myLabel['behavior'])
behaviornon1 = np.asarray(myLabel['behaviornon1'])
time = np.asarray(myLabel['time'])
condition = np.asarray(myLabel['condition'])
N = len(mouse)


In [ ]:
X = np.hstack((power,coherence,granger))


In [ ]:
N = len(mouse)

In [ ]:
indx_pos = (behaviornon1==1)&(condition==4)
indx_neg1 = (behaviornon1==2)&(condition==4)
indx_neg2 = (behaviornon1==2)&(condition==6)
indx_neg3 = (behaviornon1==2)&(condition==8)
indx_neg = indx_neg1|indx_neg2|indx_neg3
indx_tot = indx_neg|indx_pos


In [ ]:
y = np.zeros(N)
y[indx_pos] = 1 

mouse = mouse[indx_tot]
group = group[indx_tot]
expDate = expDate[indx_tot]
behavior = behavior[indx_tot]
behaviornon1 = behaviornon1[indx_tot]
time = time[indx_tot]
condition = condition[indx_tot]
y = y[indx_tot]

N = len(mouse) 

X = np.hstack((power,coherence,granger))
X = X[indx_tot]




In [ ]:
X.shape

In [ ]:
power,coherence,granger,labels_new = load_data('/media/austin/ThickBoy__1/DataAgression_Granger2/CL_baseline_all_validate3.mat',fBounds=(1,56),feature_list=['power','coherence','granger'])


In [ ]:
power = 10*power
power = power.astype(np.float32)
power[power>6] = 6

coherence = coherence.astype(np.float32)
granger = np.exp(granger)
granger[granger>10] = 10
granger = granger.astype(np.float32)

X_new = np.hstack((power,coherence,granger))


In [ ]:
windows_new = labels_new['windows']
mouse_new = np.squeeze(windows_new['mouse'])
expDate_new = np.squeeze(windows_new['expDate'])
group_new = np.squeeze(windows_new['group'])
condition_new = np.squeeze(windows_new['condition'])
behavior_new = np.squeeze(windows_new['behavior'])
time_new = np.squeeze(windows_new['time'])

idx_pos_new = (condition_new==4)&(behavior_new==1)
indx_neg_new = (behavior_new==2)&((condition_new==4)|(condition_new==6)|(condition_new==8))
y_new = np.zeros(len(mouse_new))
y_new[idx_pos_new] = 1
idx_tot_new = idx_pos_new|indx_neg_new

X_new = X_new[idx_tot_new]
mouse_new = mouse_new[idx_tot_new]
y_new = y_new[idx_tot_new]


## Now split the mice

In [ ]:
from data_aux import split_mouse_idx

In [ ]:
len(np.unique(mouse))

In [ ]:
len(np.unique(mouse_new))

In [ ]:
ids_19,_ = split_mouse_idx(mouse,10)
ids_8,_ = split_mouse_idx(mouse_new,4)

In [ ]:
X_train_old = X[ids_19]
X_train_new = X_new[ids_8]
X_train = np.vstack((X_train_old,X_train_new))

X_test_old = X[ids_19==False]
X_test_new = X_new[ids_8==False]
X_test = np.vstack((X_test_old,X_test_new))

In [ ]:
y_train_old = np.zeros(X_train_old.shape[0])
y_train_new = np.ones(X_train_new.shape[0])
y_train = np.concatenate((y_train_old,y_train_new))

y_test_old = np.zeros(X_test_old.shape[0])
y_test_new = np.ones(X_test_new.shape[0])
y_test = np.concatenate((y_test_old,y_test_new))

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
model_nmf = dp.NMF(30)
S_train = model_nmf.fit_transform(X_train)
S_test = model_nmf.transform(X_test)

In [ ]:
X_recon_train = np.dot(S_train,model_nmf.components_)
X_recon_test = np.dot(S_test,model_nmf.components_)
resid_train = X_train-X_recon_train
resid_test = X_test-X_recon_test

In [ ]:
mod_lr = LR(C=.001)
mod_lr.fit(resid_train,y_train)
y_pred = mod_lr.decision_function(resid_test)
print(roc_auc_score(y_test,y_pred))

In [ ]:
fpr,tpr,_ = roc_curve(y_test,y_pred)
plt.plot(fpr,tpr)